---
title: "Conflict-Free Replicated Data Types"
version: 2
abstract: >
    FIXME
syllabus:
-   Explain why last-write-wins registers can lose updates in concurrent systems and when that trade-off is acceptable.
-   Implement a grow-only counter and a positive-negative counter using the merge-by-maximum rule.
-   Describe the add-wins semantics of an OR-Set and trace through a concurrent add-and-remove to show the correct outcome.
-   Explain what "convergence" means for a replicated data structure and why it holds even after a network partition.
---

**In Development**

When multiple people edit an online document simultaneously,
the system ensures everyone eventually sees the same content.
Similarly,
when one person edits a document offline and the reconnects,
their changes are merged into the online version.
Traditional approaches to managing this require locking or complex transformations.
[%g crdt "Conflict-Free Replicated Data Types" %] (CRDTs),
on the other hand,
are designed so that concurrent updates on different replicas
can always be merged automatically without conflicts,
and all replicas eventually converge to the same state.
Updates are always accepted immediately,
and no locking or consensus protocol is needed.

CRDTs guarantee [%g strong-eventual-consistency "strong eventual consistency" %],
which means three things:

1.  Eventual delivery: every update reaches every replica eventually.
1.  Convergence: replicas that have received the same updates are in the same state.
1.  No conflicts: concurrent updates can always be merged automatically.

The insight that CRDTs rely on
is that some operations are [%g commutativity "commutative" %] and [%g associativity "associative" %].
The first property means that order doesn't matter,
i.e., that A+B = B+A.
The second means that group doesnt' matter,
so (A+B)+C = A+(B+C).
If the merge operation for the data type has these properties,
replicas can receive updates in any order and still converge to the same state.
Some approaches to implementing CRDTs also require operations to be [%g idempotence "idempotent" %],
which means that the operation can be applied any number of times
with the same cumulative effect
(just as zero can be added to a number over and over).

There are two approaches to building CRDTs.
In a [%g state-based-crdt "state-based CRDT" %],
replicas send their entire state and merge those states.
State-based CRDTs are simpler to reason about,
but have higher network cost because the entire state must be sent for each operation.
They also require merge operations to be commutative, associate, *and* idempotent.

In contrast,
the replicas in [%g op-based-crdt "operation-based CRDTs" %] send each other *changes* in state
(sometimes called [%g delta "deltas" %]).
This reduces the network overhead,
but requires exactly-once delivery of operations.
Those operations must commute,
but needn't be idempotent.
We will implement both approaches to understand the trade-offs.

## Last-Write-Wins Register {: #dxdx-lww}

Let's start with the simplest CRDT:
a [%g lww-register "last-write-wins register" %].
For values that should be overwritten (like a user's profile name),
we can use timestamps to determine which write wins.

[%inc lwwregister.py mark=lww %]

To see how this works,
consider three replicas that randomly choose color values to share with peers:

[%inc ex_lwwregister.py mark=replica %]

If we run three replicas for 10 timesteps,
the output is:

[%inc ex_lwwregister.out %]

Chiti's final value is different because all three replicas wrote at time 10:
Ahmed and Baemi set "red", while Chiti set "green".
Since they all have the same timestamp,
the register breaks ties by comparing replica IDs.
If the simulation ran a little longer,
all three copies would converg on "green"
because Chiti > Baemi > Ahmed alphabetically.

An LWW-Register has a weakness:
concurrent writes to the same register result in one being lost
(either the one with the earlier timestamp or with the lower replica ID).
This is acceptable for some use cases (like "last edit wins" in a profile),
but not for others.
The key trade-off is simplicity versus data preservation.
An LWW register never produces a conflict that needs manual resolution,
but it also never preserves both sides of a concurrent update.
This makes it a poor fit for situations where losing a write is costly,
such as a shared to-do list where two people add items at the same time.

## Counters {: #crdt-counter}

Another CRDT is a [%g grow-only-counter "grow-only counter" %]
whose value can only increase.
Each replica maintains a vector of counters, one per replica:

[%inc gcounter.py mark=gcounter %]

The G-Counter works by having each replica only modify its own entry in the vector.
When merging,
we take the maximum of each entry.
This operation is commutative, associative, and idempotent,
which guarantees convergence.

A G-Counter solves the problem of counting across multiple replicas that can't always communicate.
Imagine three servers tracking how many times a button has been clicked.
Users hit whichever server is nearest, and the servers sync up when they can.
A naive approach is to have each server keep a single integer and share its value with the other two,
but this breaks because there's no way to tell whether a value Ahmed receives from Baemi
already includes Ahmed's updates or not.

In a G-Counter,
each replica only increments its own entry:
for example,
Ahmed's server only ever touches `counts["Ahmed"]`.
This ensures that there's never a conflict because no two replicas write to the same slot.
When replicas sync,
they can safely take the `max` of each slot,
because a higher value always means that replica has done more increments.
`max` is idempotent (applying the same sync twice is harmless),
commutative (order doesn't matter),
and associative (grouping doesn't matter),
which makes it suitable for a CRDT.

It's important to note that the overall value of the G-Counter is the sum of the individual values,
not their maximum or any single local value.
Again,
each slot tracks how many increments that specific replica has performed,
so the total is how many increments have happened across all replicas.

To make it concrete, consider this output:

```txt
Ahmed: value=5, counts={'Ahmed': 3, 'Baemi': 2}
Baemi: value=5, counts={'Baemi': 3, 'Ahmed': 2}
Chiti: value=7, counts={'Chiti': 3, 'Ahmed': 2, 'Baemi': 2}
```

Ahmed and Baemi have synced with each other,
so they agree,
but neither has synced recently with Chiti.
Chiti has synced with them, though,
so Chiti's view is more complete.
Once all three sync,
they'll all converge to:

```txt
counts={'Ahmed': 3, 'Baemi': 3, 'Chiti': 3}`
```

with the value 9.
No increments are lost, regardless of sync order or timing.

A grow-only counter is limited.
What if we want to be able to decrement the value?
A [%g pn-counter "positive-negative counter" %], or PN-Counter, uses two G-Counters:
one for increments, one for decrements.
This works because increments and decrements are tracked separately.
Each remains monotonically increasing, so the G-Counter merge properties still apply.

[%inc pncounter.py mark=pncounter %]

## Observed-Remove Set (OR-Set) {: #crdt-orset}

Sets are trickier to implement than counters.
If Ahmed adds X and Baemi removes X concurrently, should X be in the final set?

The most natural answer is "remove wins": the element is absent.
But this makes it impossible to add an element back after a concurrent remove has been received—
the remove has already been applied.
The OR-Set chooses "add-wins" semantics instead,
by using unique tags to distinguish each add operation.

When Ahmed adds element X, the OR-Set creates a unique tag (say, `(Ahmed, 42)`) and records the pair `(X, {(Ahmed,42)})`.
When Baemi later removes X, the remove records *which tags it has seen*—in this case `(Ahmed, 42)`.
If Chiti concurrently adds X with a new tag `(Chiti, 7)`, that tag has never been seen by Baemi's remove.
After merging all three replicas, X is still present because tag `(Chiti, 7)` was added but never removed.
The element is present if and only if there exists at least one add tag that no remove has ever seen.

To make the concurrent case concrete:

```
Time 0: all replicas have {}

Ahmed adds X → Ahmed's state: {X: {(Ahmed,1)}}
Baemi removes X → Baemi observes {X: {(Ahmed,1)}}, so removes tag (Ahmed,1)
                   Baemi's state: {} (no remaining tags for X)
Chiti adds X  → (concurrent with Baemi's remove)
                   Chiti's state: {X: {(Chiti,1)}}

Merge all three:
  Ahmed: {X: {(Ahmed,1)}}
  Baemi: {} removes={(Ahmed,1)}
  Chiti: {X: {(Chiti,1)}}
  Result: X is present because (Chiti,1) was never removed.
```

[%inc orset.py mark=orset %]

This example shows an OR-set in operation:

[%inc ex_orset.py mark=replica %]
[%inc ex_orset.out %]

## Operation-Based CRDTs {: #crdt-op}

While state-based CRDTs send the full state between replicas
operation-based CRDTs send just the operations.
Let's implement an operation-based counter
by defining a dataclass to represent a single operation:

[%inc opbased_counter.py mark=op %]

and then a class to use it:

[%inc opbased_counter.py mark=counter %]

Operation-based CRDTs require reliable broadcast
to ensure that every operation reaches every replica exactly once.
In practice,
this means tracking which operations have been delivered and handling duplicates,
which is what the `applied_ops` member of the `OpBasedCounter` class above does.

"Exactly once" does not mean the network must guarantee it—networks never do.
Instead, the sender can re-send operations as many times as needed (achieving "at least once"),
and the receiver deduplicates using the unique operation ID (`applied_ops`).
Together these give "exactly once" *effect*: the operation changes state only the first time it arrives.
This is why operation IDs must be globally unique (not just unique per replica) and must be stored permanently—
discarding `applied_ops` would allow a re-delivered operation to be applied twice, breaking convergence.

The `Replica` class below exercises this counter:

[%inc ex_opbased_counter.py mark=replica %]

Unlike the state-based examples that sync by merging full state,
each replica creates an increment or decrement operation with a unique ID,
applies it locally,
and then broadcasts it to all the other replicates.
The replica then drains its inbox and applies received operations,
skipping duplicates via `op_id`.
As the output below shows,
all replicas converge to the same value
because every operation is delivered to every replica exactly once:

[%inc ex_opbased_counter.out %]

## Network Partition Simulation {: #crdt-partition}

A [%g network-partition "network partition" %] happens
when nodes in a distributed system temporarily can't communicate,
which causes them to form isolated groups.
As a result,
messages sent from one part of the system may not reach another,
effectively splitting the system into disconnected segments.

One of CRDTs' key benefits is [%g partition-tolerance "partition tolerance" %].
Let's simulate a network partition using the `GCounter` class defined earlier.
First,
we create a simple dataclass to represent peers in the network:

[%inc ex_partition.py mark=peer %]

Next,
we define a `Replica` process that repeatedly tries to synchronize
with a randomly-selected peer:

[%inc ex_partition.py mark=replica %]

We then create a partition controller that creates and heals a partition:

[%inc ex_partition.py mark=partition %]

Finally,
we create three replicas and force a break in the network
at a particular time and for a particular duration:

[%inc ex_partition.py mark=sim %]

As the output shows,
the counter recovers from the partitioning:

[%inc ex_partition.out %]

<section class="exercises" markdown="1">

## Exercises {: #crdt-exercises}

1.  The LWW-Register breaks ties by comparing replica IDs alphabetically.
    Change the simulation so that Chiti's replica ID is "AAA" (alphabetically first).
    What is the final value after merging?
    Why might choosing the alphabetically last ID as the winner be a bad choice for real deployments?

2.  In the G-Counter, each replica only modifies its own slot in the vector.
    What happens if two replicas accidentally use the same replica ID?
    Trace through a scenario with two replicas both named "R1" doing one increment each and then merging.
    What is the final value, and why is it wrong?

3.  The OR-Set gives "add-wins" semantics.
    Design an "remove-wins" set where a concurrent add and remove results in the element being absent.
    Sketch the data structure:
	you do not need to implement it fully,
	just describe what the merge operation must do differently.

4.  In the operation-based counter, `applied_ops` grows without bound.
    Propose a strategy to bound its size.
    What guarantee must hold for your strategy to be safe?
    (Hint: think about what it means for all replicas to have received an operation.)

5.  In the network partition simulation, the counter recovers after the partition heals.
    Run the simulation with the partition lasting twice as long.
    Does the final value change?
    Now run it with the partition never healing.
    What happens, and does this violate any of the three CRDT guarantees?

</section>